# Selezione Dinamica della Dimensione del Campione — Raccomandazione Musicale con Feedback Implicito

Notebook di riproduzione di un **nuovo esperimento applicativo** della tesi
*"Selezione Dinamica della Dimensione del Campione in Metodi di Ottimizzazione
per il Machine Learning"*: la **raccomandazione musicale su larga scala** con
feedback implicito (ascolti), modellata come **Logistic Matrix Factorization**
(regressione logistica pesata su tutte le coppie utente–artista).

Il notebook:

1. scarica il dataset **Last.fm (HetRec 2011)** — ascolti utente→artista
   (`user_artists.dat`),
2. costruisce la **griglia completa** delle N = n_u × n_i coppie
   (utente, artista) con peso di confidenza c_ui = 1 + α·r̂_ui (i mancati
   ascolti sono esempi negativi di peso 1, come in Hu et al. 2008 e
   Johnson 2014),
3. esegue i **quattro metodi della tesi** — codice **esattamente come nei
   listati B.1–B.4 dell'Appendice B** — sulla regressione logistica pesata,
   una funzione **fortemente convessa** (regolarizzazione L2),
4. valuta il ranking delle raccomandazioni (Recall@10, NDCG@10) con la
   **baseline popolarità** come riferimento,
5. mostra le **predizioni dei 4 modelli** su un utente esempio,
6. rigenera **figure e tabelle** (o salva i dati per rigenerarle).

> **Riproducibilità**: seme fissato (`seed = 42`), risultati deterministici.
> Se il download del dataset fallisce (nessuna rete), viene generato un
> dataset sintetico con lo stesso formato, così il notebook resta eseguibile.


In [ ]:
#@title 0. Dipendenze e import
# pandas e' gia' presente in Colab; il comando e' solo un safeguard.
%pip install -q pandas

import os, json, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("numpy ", np.__version__)
print("pandas", pd.__version__)


In [ ]:
#@title 1. Scarica il dataset Last.fm (HetRec 2011)
# user_artists.dat: tripla  userID  artistID  plays  (ascolti impliciti).
# Fonte: https://grouplens.org/datasets/hetrec-2011/

URL_ZIP = "https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip"
ZIP = "hetrec2011-lastfm-2k.zip"
F_USER_ART, F_ARTISTS = "user_artists.dat", "artists.dat"

def scarica_e_apri():
    if not os.path.exists(F_USER_ART):
        ok = False
        for u in (URL_ZIP, URL_ZIP.replace("https:", "http:")):
            try:
                print("Scaricamento", u, "...")
                urllib.request.urlretrieve(u, ZIP)
                ok = True
                break
            except Exception as e:  # noqa: BLE001
                print("  fallito:", e)
        if not ok:
            raise RuntimeError("download non riuscito")
        with zipfile.ZipFile(ZIP) as z:
            z.extract(F_USER_ART)
            z.extract(F_ARTISTS)
    df  = pd.read_csv(F_USER_ART, sep="\t")
    art = pd.read_csv(F_ARTISTS,  sep="\t")
    df.columns = ["userID", "artistID", "plays"]
    return df, art

try:
    df, artists_df = scarica_e_apri()
    print("coppie (utente, artista):", len(df))
    print(df.head())
    SYNTHETIC = False
except Exception as e:  # noqa: BLE001
    print("Dataset non disponibile (", e, ") -> si usano dati SINTETICI.")
    SYNTHETIC = True


In [ ]:
#@title 2. Costruzione del problema (griglia completa + split val/test)
# In feedback implicito l'obiettivo somma su TUTTI gli N = n_u*n_i
# accoppiamenti (utente, artista): quelli osservati (peso alto) e quelli
# non osservati (peso 1). Ogni accoppiamento e' un "esempio" per i listati
# B.1-B.4: l'indice i codifica la coppia (u = i//n_i, v = i%n_i).

SEED = 42
N_U, N_I, K = 250, 200, 8      # utenti, artisti, fattori latenti (modifica pure)
ALPHA_C = 40.0                 # peso osservati: c = 1 + ALPHA_C * r_hat
VAL_FRAC, TEST_FRAC = 0.1, 0.1 # frazione di positivi per utente tenuti fuori
np.random.seed(SEED)

if not SYNTHETIC:
    # sottodominio denso: utenti piu' attivi e artisti piu' ascoltati
    sub = df[df["userID"].isin(df.groupby("userID")["plays"].sum().nlargest(N_U).index)]
    sub = sub[sub["artistID"].isin(sub.groupby("artistID")["plays"].sum().nlargest(N_I).index)]
    act = sub.groupby("userID")["plays"].sum()
    u_ids = sorted(act[act > 0].index)
    v_ids = sorted(sub["artistID"].unique())
    u_map = {u: i for i, u in enumerate(u_ids)}
    v_map = {a: i for i, a in enumerate(v_ids)}
    n_u, n_i = len(u_map), len(v_map)
    PLAYS = np.zeros((n_u, n_i), dtype=float)
    for _, row in sub.iterrows():
        PLAYS[u_map[row.userID], v_map[row.artistID]] = row.plays
    art_names = dict(zip(artists_df["id"], artists_df["name"]))
    ART_NAMES = [art_names.get(a, str(a)) for a in v_ids]
else:
    # dataset sintetico: fattori latenti veri + ascolti bernoulliani
    n_u, n_i = N_U, N_I
    rng = np.random.default_rng(SEED)
    Us = rng.normal(0, 0.3, (n_u, K)); Vs = rng.normal(0, 0.3, (n_i, K))
    bus = rng.normal(0, 0.1, n_u);     bvs = rng.normal(0, 0.1, n_i)
    Pz = 1.0 / (1.0 + np.exp(-(Us @ Vs.T + bus[:, None] + bvs[None, :])))
    PLAYS = rng.binomial(1, np.clip(Pz, 0, 1)).astype(float)
    ART_NAMES = [f"Artista {i}" for i in range(n_i)]

N = n_u * n_i
print(f"utenti={n_u}  artisti={n_i}  fattori={K}  N={N}  parametri={n_u*K+n_i*K+n_u+n_i}")

# --- target su tutta la griglia (positivi = ascolto osservato) ---
P_all = (PLAYS > 0).astype(float)

# --- split val/test dei positivi, per utente (stream indipendente dal train) ---
VAL = np.zeros_like(P_all, bool)
TEST = np.zeros_like(P_all, bool)
for u in range(n_u):
    pos = np.where(P_all[u] > 0)[0]
    if len(pos) == 0:
        continue
    perm = np.random.default_rng(SEED + u).permutation(pos)
    nv = min(max(int(round(VAL_FRAC * len(pos))), 1), len(pos))
    nt = min(max(int(round(TEST_FRAC * len(pos))), 1), len(pos) - nv)
    VAL[u, perm[:nv]] = True
    TEST[u, perm[nv:nv + nt]] = True

# --- training: i positivi val/test tornano "non osservati" ---
PL_tr = PLAYS.copy()
PL_tr[VAL | TEST] = 0.0
rhat_tr = np.minimum(1.0, PL_tr / max(1.0, np.percentile(PL_tr[PL_tr > 0], 90)))
CU = 1.0 + ALPHA_C * rhat_tr      # matrice dei pesi c_ui (usata dai listati)
PU = (PL_tr > 0).astype(float)    # matrice dei target p_ui
print("positivi in train:", int(PU.sum()),
      " | in val:", int(VAL.sum()),
      " | in test:", int(TEST.sum()))


## Il problema di apprendimento: Logistic Matrix Factorization

Dati n_u utenti e n_i artisti, sia r_ui il numero di ascolti (feedback
implicito) e p_ui = 1 se r_ui > 0. La **Logistic Matrix Factorization**
(Johnson 2014) minimizza la regressione logistica pesata su **tutte le
coppie**:

J(w) = (1/N) Σ_{u,i} c_ui [ log(1 + e^{z_ui}) − p_ui z_ui ] + (λ/2)(‖U‖²_F + ‖V‖²_F)

con z_ui = u·v_i + b_u + b_v, c_ui = 1 + α·r̂_ui, e w = vec(U, V, b_u, b_v).
Il peso c_ui fa sì che gli ascolti osservati contino molto più dei mancati
ascolti (che restano comunque esempi, con c=1): il modello "scopre" la
struttura latente dei gusti.

Con λ > 0 l'obiettivo è **fortemente convesso** (somma di perdite logistiche
convesse + termine quadratico): siamo esattamente nelle ipotesi della tesi
(Sez. 3.5) e i quattro metodi convergono al minimo globale.

I listati B.1–B.4 operano su un generico vettore w attraverso le funzioni
globali `N`, `loss_i(w, i)`, `grad_i(w, i)`, `hessvec_i(w, i, v)` e
`grad_full(w)`. **Gli algoritmi (celle successive) sono copiati letteralmente
dai listati e non vengono modificati.**

La regolarizzazione L2 (λ = 10⁻²) è inclusa in `loss_i`/`grad_i`/`hessvec_i`
per Dynamic GD, Newton-CG e BB-CCV; per Newton-CG L1 si imposta λ = 0 (la
penalità L1 è gestita dall'algoritmo stesso con ν = 10⁻³).


In [ ]:
#@title 3. Interfaccia usata dai listati B.1-B.4 (specifica del problema)
# Ogni indice i = 0..N-1 codifica la coppia (u = i//n_i, v = i%n_i).
# loss_i/grad_i/hessvec_i sono l'interfaccia "dipendente dal dataset" che i
# listati dell'Appendice B assumono come gia' definite (per-esempio).

D = n_u * K + n_i * K + n_u + n_i      # dimensione di w = vec(U, V, b_u, b_v)
N = n_u * n_i
LAM = 1e-2                             # regolarizzazione L2 (0 per il metodo L1)
R = 0.1                                # rapporto |H_k| / |S_k| (usato in B.2, B.3)

_wcache = {"w": None, "parts": None}
def _parts(w):
    if _wcache["w"] is not w:
        U  = w[:n_u*K].reshape(n_u, K)
        V  = w[n_u*K:n_u*K+n_i*K].reshape(n_i, K)
        bu = w[n_u*K+n_i*K:n_u*K+n_i*K+n_u]
        bv = w[n_u*K+n_i*K+n_u:]
        _wcache["w"] = w
        _wcache["parts"] = (U, V, bu, bv)
    return _wcache["parts"]

def _sig(z):
    return 1.0 / (1.0 + np.exp(np.clip(-z, -745, 745)))

def loss_i(w, i):
    U, V, bu, bv = _parts(w)
    u, v = i // n_i, i % n_i
    z = U[u] @ V[v] + bu[u] + bv[v]
    c, p = CU[u, v], PU[u, v]
    data = c * (np.logaddexp(0.0, z) - p * z)
    reg  = 0.5 * LAM * (U[u] @ U[u] + V[v] @ V[v])
    return float(data) + reg

def grad_i(w, i):
    U, V, bu, bv = _parts(w)
    u, v = i // n_i, i % n_i
    z = U[u] @ V[v] + bu[u] + bv[v]
    c, p = CU[u, v], PU[u, v]
    d = c * (_sig(z) - p)
    g = np.zeros(D)
    g[u*K:(u+1)*K] = d * V[v] + LAM * U[u]
    g[n_u*K+v*K:n_u*K+(v+1)*K] = d * U[u] + LAM * V[v]
    g[n_u*K+n_i*K+u] = d
    g[n_u*K+n_i*K+n_u+v] = d
    return g

def hessvec_i(w, i, pt):
    U, V, bu, bv = _parts(w)
    U1, V1, bu1, bv1 = _parts(pt)
    u, v = i // n_i, i % n_i
    z = U[u] @ V[v] + bu[u] + bv[v]
    c, p = CU[u, v], PU[u, v]
    sig = _sig(z)
    s = sig * (1.0 - sig)
    du, dv = U1[u], V1[v]
    dz = du @ V[v] + U[u] @ dv + bu1[u] + bv1[v]
    h = c * s * dz
    Hv = np.zeros(D)
    Hv[u*K:(u+1)*K] = h * V[v] + c * (sig - p) * dv + LAM * du
    Hv[n_u*K+v*K:n_u*K+(v+1)*K] = h * U[u] + c * (sig - p) * du + LAM * dv
    Hv[n_u*K+n_i*K+u] = h
    Hv[n_u*K+n_i*K+n_u+v] = h
    return Hv

def grad_full(w):                     # vettorizzato su tutta la griglia
    U, V, bu, bv = _parts(w)
    Z = U @ V.T + bu[:, None] + bv[None, :]
    P = _sig(Z)
    Dm = CU * (P - PU)
    gU  = (Dm @ V) / N + LAM * U
    gV  = (Dm.T @ U) / N + LAM * V
    gbu = Dm.sum(axis=1) / N
    gbv = Dm.sum(axis=0) / N
    return np.concatenate([gU.ravel(), gV.ravel(), gbu, gbv])

def loss_full(w):
    U, V, bu, bv = _parts(w)
    Z = U @ V.T + bu[:, None] + bv[None, :]
    data = (CU * (np.logaddexp(0.0, Z) - PU * Z)).mean()
    reg  = 0.5 * LAM * (np.sum(U**2) + np.sum(V**2))
    return float(data) + reg


## I quattro algoritmi della tesi (Appendice B)

Le celle seguenti contengono **integralmente il codice dei listati B.1–B.4
dell'Appendice B della tesi** (Dynamic GD, Newton-CG, Newton-CG $L_1$,
BB-CCV), copiato senza modifiche. Gli algoritmi operano sulle funzioni
globali `N`, `loss_i`, `grad_i`, `hessvec_i`, `grad_full` definite nella
cella precedente: **non è necessario modificare nulla** per passare dal
problema della tesi al problema di raccomandazione musicale.


#### Listato B.1 — Dynamic GD (CCV + line search di Wolfe)


In [ ]:
import numpy as np

def dynamic_gd(w0, theta, max_iter, alpha, batch0):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history     = [w.copy().tolist()]
    batch_sizes = [n]
    for k in range(max_iter):
        indices = np.random.choice(N, size=n, replace=False)
        grads = np.array([grad_i(w, i) for i in indices])
        g = np.mean(grads, axis=0)

        if n > 1:
            var_vec = np.var(grads, axis=0, ddof=1)
        else:
            var_vec = np.zeros_like(g)
        V_norm1 = np.sum(var_vec)

        gg = np.dot(g, g)
        if gg > 1e-16:
            if V_norm1 / n > theta**2 * gg:
                n_new = int(np.ceil(V_norm1 / (theta**2 * gg))) + 1
                n = min(n_new, N)

        def J_batch(w_curr):
            return np.mean([loss_i(w_curr, i) for i in indices])

        # Line search di Wolfe
        c1, c2 = 1e-4, 0.9
        step = alpha
        J_curr = J_batch(w)
        g_norm2 = np.dot(g, g)
        d = -g
        gd = -g_norm2
        if g_norm2 > 1e-16:
            for _ in range(30):
                w_new = w + step * d
                if J_batch(w_new) <= J_curr + c1 * step * gd:
                    g_new = np.mean([grad_i(w_new, i) for i in indices], axis=0)
                    if np.dot(g_new, d) >= c2 * gd:
                        break
                step *= 0.5
            else:
                step = 0.0

        w = w + step * d

        if np.linalg.norm(grad_full(w)) < 1e-6:
            history.append(w.copy().tolist())
            batch_sizes.append(n)
            break

        history.append(w.copy().tolist())
        batch_sizes.append(n)

    return history, batch_sizes


#### Listato B.2 — Newton-CG (Hessiana sottocampionata + CG adattivo)


In [ ]:
import numpy as np

def cg(A, b, gamma, maxcg):
    x = np.zeros_like(b)
    r = b - A(x)
    p = r.copy()
    rr = np.dot(r, r)
    for _ in range(maxcg):
        Ap = A(p)
        pHp = np.dot(p, Ap)
        if pHp <= 1e-14:
            break
        alpha = rr / pHp
        x = x + alpha * p
        r_new = r - alpha * Ap
        rr_new = np.dot(r_new, r_new)
        if rr_new <= gamma * np.dot(x, x) + 1e-16:
            return x
        beta = rr_new / rr
        p = r_new + beta * p
        r = r_new
        rr = rr_new
    return x

def newton_cg(w0, theta, max_iter, alpha, batch0, R, maxcg):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history, batch_sizes = [w.copy().tolist()], [n]
    for k in range(max_iter):
        indices_S = np.random.choice(N, size=n, replace=False)
        g = np.mean([grad_i(w, i) for i in indices_S], axis=0)
        n_h = min(max(1, int(round(R * n))), N)
        indices_H = np.random.choice(indices_S, size=n_h, replace=False)
        Hv = lambda v: np.mean([hessvec_i(w, i, v) for i in indices_H], axis=0)
        p0 = -g
        p0_norm2 = np.dot(p0, p0)
        gamma = 0.0
        if p0_norm2 > 1e-16 and n_h > 1:
            Hp0 = np.array([hessvec_i(w, i, p0) for i in indices_H])
            gamma = np.sum(np.var(Hp0, axis=0, ddof=1)) / (n_h * p0_norm2)
        d = cg(Hv, -g, gamma, maxcg)
        J_batch = lambda wc: np.mean([loss_i(wc, i) for i in indices_S])

        c1, c2 = 1e-4, 0.9
        step, J_w = alpha, J_batch(w)
        gd = np.dot(g, d)
        if gd >= 0:
            d = -g
            gd = -np.dot(g, g)
        for _ in range(30):
            w_new = w + step * d
            if J_batch(w_new) <= J_w + c1 * step * gd:
                g_new = np.mean([grad_i(w_new, i) for i in indices_S], axis=0)
                if np.dot(g_new, d) >= c2 * gd:
                    break
            step *= 0.5
        else:
            step = 0.0
        w = w + step * d
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        indices_new = np.random.choice(N, size=n, replace=False)
        g_new = np.mean([grad_i(w, i) for i in indices_new], axis=0)
        var_vec = np.var([grad_i(w, i) for i in indices_new], axis=0, ddof=1) if n > 1 else np.zeros_like(g_new)
        V_norm1, gg_new = np.sum(var_vec), np.dot(g_new, g_new)
        if gg_new > 1e-16 and V_norm1 / n > theta**2 * gg_new:
            n = min(int(np.ceil(V_norm1 / (theta**2 * gg_new))) + 1, N)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes


#### Listato B.3 — Newton-CG per problemi $L_1$-regolarizzati


In [ ]:
import numpy as np

def newton_l1(w0, theta, max_iter, alpha, batch0, nu, sigma, maxcg, eta=0.5):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history     = [w.copy().tolist()]
    batch_sizes = [n]
    def F_batch(v, indices):
        Jb = np.mean([loss_i(v, i) for i in indices])
        return Jb + nu * np.sum(np.abs(v))
    def subgrad_batch(v, indices):
        grads = np.array([grad_i(v, i) for i in indices])
        gJ = np.mean(grads, axis=0)
        g = np.zeros_like(v)
        for i in range(len(v)):
            if v[i] > 0:
                g[i] = gJ[i] + nu
            elif v[i] < 0:
                g[i] = gJ[i] - nu
            else:
                if gJ[i] < -nu:
                    g[i] = gJ[i] + nu
                elif gJ[i] > nu:
                    g[i] = gJ[i] - nu
                else:
                    g[i] = 0.0
        return g
    def project_orthant(v, z):
        res = v.copy()
        for i in range(len(v)):
            if z[i] != 0 and np.sign(res[i]) != z[i]:
                res[i] = 0.0
        return res
    for k in range(max_iter):
        indices_S = np.random.choice(N, size=n, replace=False)
        grads = np.array([grad_i(w, i) for i in indices_S])
        g_batch = np.mean(grads, axis=0)
        z = np.where(w > 0, 1,
            np.where(w < 0, -1,
                np.where(g_batch < -nu, 1,
                    np.where(g_batch > nu, -1, 0))))
        sg = subgrad_batch(w, indices_S)
        sgn = np.linalg.norm(sg)
        if sgn < 1e-10:
            history.append(w.copy().tolist())
            batch_sizes.append(n)
            break
        n_h = max(1, int(round(R * n)))
        n_h = min(n_h, N)
        indices_H = np.random.choice(indices_S, size=n_h, replace=False)
        free = (z != 0)
        d = np.zeros_like(w)
        if np.any(free):
            g_free = sg[free]
            # Hessian free: prodotto H·v senza costruire H
            def Hv(v_full):
                return np.mean([hessvec_i(w, i, v_full) for i in indices_H], axis=0)
            def Hv_free(v_free):
                v_full = np.zeros_like(w)
                v_full[free] = v_free
                Hv_full = Hv(v_full)
                return Hv_full[free]
            tol_cg = eta * np.linalg.norm(g_free)
            d_free = np.zeros(np.sum(free))
            r = -g_free.copy()
            p = r.copy()
            rr = np.dot(r, r)
            for _ in range(maxcg):
                Hp = Hv_free(p)
                pHp = np.dot(p, Hp)
                if pHp <= 1e-14:
                    if np.linalg.norm(d_free) < 1e-14:
                        d_free = -g_free.copy()
                    break
                alpha_cg = rr / pHp
                d_free = d_free + alpha_cg * p
                r_new = r - alpha_cg * Hp
                rr_new = np.dot(r_new, r_new)
                if np.sqrt(rr_new) <= tol_cg:
                    r = r_new
                    rr = rr_new
                    break
                beta = rr_new / rr
                p = r_new + beta * p
                r = r_new
                rr = rr_new
            d[free] = d_free
        step = alpha
        F_w = F_batch(w, indices_S)
        sg_d = np.dot(sg, d)
        if sg_d >= 0:
            d = -sg
            sg_d = -np.dot(sg, sg)
        w_new = w.copy()
        for _ in range(20):
            w_trial = project_orthant(w + step * d, z)
            if F_batch(w_trial, indices_S) <= F_w + sigma * step * sg_d:
                w_new = w_trial
                break
            step *= 0.5
            if step < 1e-12:
                w_new = w.copy()
                break
        w = w_new
        indices_new = np.random.choice(N, size=n, replace=False)
        grads_new = np.array([grad_i(w, i) for i in indices_new])
        g_new = np.mean(grads_new, axis=0)
        if n > 1:
            var_vec = np.var(grads_new, axis=0, ddof=1)
        else:
            var_vec = np.zeros_like(g_new)
        V_norm1 = np.sum(var_vec)
        gg_new = np.dot(g_new, g_new)
        if gg_new > 1e-16:
            if V_norm1 / n > theta**2 * gg_new:
                n_new = int(np.ceil(V_norm1 / (theta**2 * gg_new))) + 1
                n = min(n_new, N)
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes


#### Listato B.4 — BB-CCV (Barzilai–Borwein con campionamento dinamico)


In [ ]:
import numpy as np

def bb_dynamic_gd(w0, theta, max_iter, alpha, batch0):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history     = [w.copy().tolist()]
    batch_sizes = [n]

    w_prev = w.copy()
    g_prev = None
    for k in range(max_iter):
        indices = np.random.choice(N, size=n, replace=False)
        grads = np.array([grad_i(w, i) for i in indices])
        g = np.mean(grads, axis=0)
        if k > 0 and g_prev is not None:
            s = w - w_prev
            y = g - g_prev
            sy = np.dot(s, y)
            if abs(sy) > 1e-14:
                step_bb = np.dot(s, s) / sy
                step = np.clip(step_bb, alpha / 20.0, alpha * 5.0)
            else:
                step = alpha
        else:
            step = alpha

        w_prev = w.copy()
        g_prev = g.copy()
        def J_batch(w_curr):
            return np.mean([loss_i(w_curr, i) for i in indices])

        c1 = 1e-4
        J_curr = J_batch(w)
        g_norm2 = np.dot(g, g)

        if g_norm2 > 1e-16:
            for _ in range(30):
                w_new = w - step * g
                if J_batch(w_new) <= J_curr - c1 * step * g_norm2:
                    break
                step *= 0.5
            else:
                step = 0.0

        w = w - step * g
        if n > 1:
            var_vec = np.var(grads, axis=0, ddof=1)
        else:
            var_vec = np.zeros_like(g)
        V_norm1 = np.sum(var_vec)

        gg = np.dot(g, g)
        if gg > 1e-16:
            if V_norm1 / n > theta**2 * gg:
                n_new = int(np.ceil(V_norm1 / (theta**2 * gg))) + 1
                n = min(n_new, N)
        history.append(w.copy().tolist())
        batch_sizes.append(n)

        if np.linalg.norm(grad_full(w)) < 1e-6:
            break

    return history, batch_sizes


In [ ]:
#@title 4. Esecuzione dei quattro metodi (listati B.1-B.4) + baseline
# Tempo stimato (Colab): ~10-20 min con MAX_ITER=200 sulla griglia 250x200.
# Per una prova rapida: MAX_ITER = 50 (e, volendo, N_U/N_I piu' piccoli
# nella cella 2).

THETA, ALPHA, BATCH0, MAX_ITER = 0.5, 1.0, 64, 200
NU, SIGMA, MAXCG = 1e-3, 1e-4, 50
w0 = np.zeros(D)

# --- valutazione del ranking (predizione) ---
def score_matrix(w):
    U, V, bu, bv = _parts(w)
    return U @ V.T + bu[:, None] + bv[None, :]

def ranking_metrics(scores, rel, K=10):
    recs, ndcgs = [], []
    for u in range(scores.shape[0]):
        pos = np.where(rel[u])[0]
        if len(pos) == 0:
            continue
        poset = set(pos.tolist())
        order = np.argsort(-scores[u])[:K]
        hits = np.isin(pos, order)
        recs.append(float(hits.sum()) / len(pos))
        dcg  = sum(1.0 / np.log2(j + 2) for j in range(len(order)) if order[j] in poset)
        idcg = sum(1.0 / np.log2(j + 2) for j in range(min(K, len(pos))))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(recs)), float(np.mean(ndcgs))

def subgrad_full_l1(w):
    gJ = grad_full(w)                      # LAM = 0 per il metodo L1
    g = np.zeros_like(w)
    for i in range(len(w)):
        if   w[i] > 0:        g[i] = gJ[i] + NU
        elif w[i] < 0:        g[i] = gJ[i] - NU
        elif gJ[i] < -NU:     g[i] = gJ[i] + NU
        elif gJ[i] > NU:      g[i] = gJ[i] - NU
        else:                 g[i] = 0.0
    return g

methods = [
    ("Dynamic GD",   lambda: dynamic_gd(w0, THETA, MAX_ITER, ALPHA, BATCH0)),
    ("Newton-CG",    lambda: newton_cg(w0, THETA, MAX_ITER, ALPHA, BATCH0, R, MAXCG)),
    ("Newton-CG L1", lambda: newton_l1(w0, THETA, MAX_ITER, ALPHA, BATCH0, NU, SIGMA, MAXCG)),
    ("BB-CCV",       lambda: bb_dynamic_gd(w0, THETA, MAX_ITER, ALPHA, BATCH0)),
]

eval_users = [u for u in range(n_u) if VAL[u].sum() > 0][:100]

results, recalls, batches, pesi = {}, {}, {}, {}
for name, run in methods:
    np.random.seed(SEED)
    LAM = 0.0 if name == "Newton-CG L1" else 1e-2
    t0 = time.time()
    history, batch_sizes = run()
    dt = time.time() - t0
    wf = np.array(history[-1])
    pesi[name] = wf.copy()
    if name == "Newton-CG L1":
        loss_f = loss_full(wf) + NU * np.sum(np.abs(wf))
        gnorm  = float(np.linalg.norm(subgrad_full_l1(wf)))
        nnz    = int(np.sum(wf != 0))
    else:
        loss_f = loss_full(wf)
        gnorm  = float(np.linalg.norm(grad_full(wf)))
        nnz    = None
    rec_test, ndcg_test = ranking_metrics(score_matrix(wf), TEST, K=10)
    results[name] = dict(iter=len(history) - 1, recall10=rec_test, ndcg10=ndcg_test,
                         loss=loss_f, gnorm=gnorm, batch=int(max(batch_sizes)),
                         batch0=batch_sizes[0], time=dt, nnz=nnz)
    recalls[name] = [ranking_metrics(score_matrix(np.array(w))[eval_users],
                                     VAL[eval_users])[0] for w in history]
    batches[name] = batch_sizes
    print(f"{name:14s} iter={results[name]['iter']:3d}  Recall@10={rec_test*100:5.2f}%  "
          f"NDCG@10={ndcg_test*100:5.2f}%  ||grad||={gnorm:.1e}  "
          f"batch_fin={results[name]['batch']:5d}  tempo={dt:5.1f}s"
          + (f"  nnz={nnz}" if nnz is not None else ""))

# --- baseline: popolarita' (ascolti totali in train) ---
pop = PL_tr.sum(axis=0)
rec_pop, ndcg_pop = ranking_metrics(np.tile(pop, (n_u, 1)), TEST, K=10)
results["Popolarità"] = dict(recall10=rec_pop, ndcg10=ndcg_pop, nnz=None)
print(f"{'Popolarità':14s}  Recall@10={rec_pop*100:5.2f}%  NDCG@10={ndcg_pop*100:5.2f}%  (baseline)")


In [ ]:
#@title 5. Figura 1 — Recall@10 (validation) vs iterazioni
COLORS = {"Dynamic GD": "#1f77b4", "Newton-CG": "#d62728",
          "Newton-CG L1": "#2ca02c", "BB-CCV": "#9467bd"}
ks = np.arange(0, MAX_ITER + 1)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
for name, _ in methods:
    r = np.asarray(recalls[name]) * 100
    ax.plot(ks[:len(r)], r, lw=1.8, label=name, color=COLORS[name])
ax.axhline(rec_pop * 100, color="gray", ls="--", lw=1.2, label="Popolarità")
ax.set_xlabel(r"Iterazione $k$")
ax.set_ylabel(r"Recall@10 sul validation (\%)")
ax.set_title("Last.fm: Recall@10 vs iterazioni")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("music_reco_recall.png", dpi=150)
fig.savefig("music_reco_recall.pdf")
plt.show()
print("Salvate: music_reco_recall.png / .pdf")


In [ ]:
#@title 6. Figura 2 — dinamica del batch n_k vs k
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for name, _ in methods:
    b = np.asarray(batches[name])
    ax.step(np.arange(len(b)), b, where="mid", lw=1.6, label=name, color=COLORS[name])
ax.set_xlabel(r"Iterazione $k$")
ax.set_ylabel(r"Dimensione del batch $n_k$")
ax.set_title("Last.fm: dinamica del batch (CCV)")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("music_reco_batch.png", dpi=150)
fig.savefig("music_reco_batch.pdf")
plt.show()
print("Salvate: music_reco_batch.png / .pdf")


In [ ]:
#@title 7. Tabella dei risultati (pronta per il LaTeX) + salvataggio dati
print("=== Tabella risultati: raccomandazione musicale (Last.fm, Logistic MF) ===")
print(r"Metodo & Recall@10 & NDCG@10 & $\|\nabla J(w)\|_2$ & Batch finale & Tempo (s) & Coeff. non nulli \\")
for name, _ in methods:
    r = results[name]
    nnz = f"{r['nnz']}/{D}" if r['nnz'] is not None else "---"
    print(f"{name} & {r['recall10']*100:.1f}\\% & {r['ndcg10']*100:.1f}\\% & "
          f"${r['gnorm']:.1e}$ & {r['batch']} & {r['time']:.1f} & {nnz} \\\\")
print(f"Popolarità (baseline) & {results['Popolarità']['recall10']*100:.1f}\\% & "
      f"{results['Popolarità']['ndcg10']*100:.1f}\\% & --- & --- & --- & --- \\\\")

# --- dati salvati (per rigenerare le tabelle fuori dal notebook) ---
_dati = {
    "problema": {"n_u": n_u, "n_i": n_i, "K": K, "N": N, "D": D,
                 "lambda": 1e-2, "nu": NU, "theta": THETA,
                 "alpha_c": ALPHA_C, "seed": SEED},
    "risultati": results,
    "note": "baseline: popolarita' (ascolti totali in train)",
}
with open("music_reco_results.json", "w", encoding="utf-8") as f:
    json.dump(_dati, f, ensure_ascii=False, indent=2)

_clean = lambda s: "".join(c if c.isalnum() else "_" for c in s)
np.savez("music_reco_curves.npz",
         **{f"recall_{_clean(k)}": np.asarray(v) for k, v in recalls.items()},
         **{f"batch_{_clean(k)}": np.asarray(v) for k, v in batches.items()})

_rows = []
for name, _ in methods:
    r = results[name]
    _rows.append({"metodo": name, "recall10": r["recall10"], "ndcg10": r["ndcg10"],
                  "grad_norm": r["gnorm"], "batch_finale": r["batch"],
                  "tempo_s": r["time"], "non_zero": r["nnz"]})
_rows.append({"metodo": "Popolarità", "recall10": results["Popolarità"]["recall10"],
              "ndcg10": results["Popolarità"]["ndcg10"]})
pd.DataFrame(_rows).to_csv("music_reco_results.csv", index=False)
print("\nSalvati: music_reco_results.json, music_reco_curves.npz, music_reco_results.csv")


In [ ]:
#@title 8. (Opzionale) Scarica figure, dati e tabelle
try:
    from google.colab import files
    for f in ("music_reco_recall.png", "music_reco_recall.pdf",
              "music_reco_batch.png", "music_reco_batch.pdf",
              "music_reco_results.json", "music_reco_curves.npz",
              "music_reco_results.csv"):
        files.download(f)
except ImportError:
    print("Non sei in Colab: i file sono nella cartella corrente.")
    print(os.listdir("."))


In [ ]:
#@title 9. (Facoltativo) Esempio utente: ascolti reali vs raccomandazioni dei 4 modelli
# Scegli un utente (indice 0..n_u-1) oppure None per uno a caso.
USER = None   # es. 12

u = int(np.random.default_rng(999).integers(n_u)) if USER is None else int(USER)
pos_test = set(np.where(TEST[u])[0].tolist())

print(f"Utente {u}  |  ascolti in train: {int(PL_tr[u].sum())}  |  positivi in test: {len(pos_test)}")

print("\nArtisti piu' ascoltati (train):")
for a in np.argsort(-PL_tr[u])[:5]:
    if PL_tr[u][a] > 0:
        print(f"  {ART_NAMES[a]:32s}  ascolti = {int(PLAYS[u][a])}")

print("\nPositivi riservati al test: " + ", ".join(ART_NAMES[a] for a in sorted(pos_test)))

print("\nTop-5 raccomandati da ciascun metodo  (✔ = nel test):")
for name, w in pesi.items():
    top = np.argsort(-score_matrix(w)[u])[:5]
    segni = "".join("✔" if int(a) in pos_test else "·" for a in top)
    print(f"  {name:14s} {segni}  " + ", ".join(ART_NAMES[a] for a in top))
